# Ingesting data into ChromaDB (the Kaizen knowledge base)

This notebook shows **how data gets into the vector DB**, what the cryptic folder
names mean, and how to add your own data.

## The 3 collections (human-readable names)
| collection | holds | fed from |
|---|---|---|
| `glayout_knowledge` | layout code + visual construction steps | `data/visual_dataset/dataset.jsonl` + `data/circuits/*/*_clean.py` |
| `rf_theory` | RF/EM/analog theory, EE-QA, PySpice | `data/rf_theory/**` |
| `error_feedback` | error → fix memory | written at runtime by the Kaizen loop |

## About those "weird number" folders in `data/chroma_db/`
They are **ChromaDB internals**, not yours to name. Each collection is stored as
**two HNSW segment folders** named by a UUID. The human-readable name → UUID map
lives in `chroma.sqlite3`. You query by the *name* (`glayout_knowledge`), never by
the folder. The cell below prints the mapping so the folders stop looking random.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('../src'))
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
from gelochip.kaizen import collections, config
import sqlite3

db = sqlite3.connect(str(config.CHROMA_DIR / 'chroma.sqlite3'))
name_of = {cid: name for name, cid in db.execute('select name, id from collections')}
print('collection name  ->  internal UUID  ->  segment folders on disk\n')
from collections import defaultdict
segs = defaultdict(list)
for sid, coll in db.execute('select id, collection from segments'):
    segs[coll].append(sid)
for cid, name in name_of.items():
    print(f'{name:20s} {cid}')
    for s in segs.get(cid, []):
        print(f'    └─ data/chroma_db/{s}/')
db.close()
print('\ncounts:', collections.collection_counts())

## 1. Ingest the glayout layout knowledge (`glayout_knowledge`)

`ingest_templates(reset=True)` rebuilds the collection from **two** sources:
1. `data/visual_dataset/dataset.jsonl` — the visual chain-of-thought dataset: one
   *construction recipe* document per circuit + one *operation* document per step
   (NL instruction + code, image path in metadata).
2. `data/circuits/*/*_clean.py` — the verified DRC-clean full circuits.

Run it after you change a clean.py or rebuild the visual dataset.

In [ ]:
# n = collections.ingest_templates(reset=True)
# print('glayout_knowledge rebuilt:', n, 'docs')
print('uncomment to rebuild glayout_knowledge')

## 2. Ingest RF/analog theory (`rf_theory`)

`ingest_theory(reset=True, parse_pdfs=False)` pulls from:
- `data/rf_theory/texts/*.txt` (pre-extracted book text)
- `data/rf_theory/arxiv/all_abstracts.txt`
- `data/rf_theory/huggingface/*.jsonl` (EE QA)
- `data/rf_theory/analog_pyspice/sft_pairs.jsonl`

**`parse_pdfs=True`** additionally OCR/parses the raw PDFs in `books/` and
`arxiv/pdfs/` (slow). By default the raw PDFs are NOT ingested — only the
pre-extracted `.txt`.

In [ ]:
# n = collections.ingest_theory(reset=True, parse_pdfs=False)
# print('rf_theory rebuilt:', n, 'docs')
print('uncomment to rebuild rf_theory (add parse_pdfs=True to include raw PDFs)')

## 3. Add a single item on the fly (no full rebuild)

- `add_template(instruction, code, circuit=...)` → adds one snippet to `glayout_knowledge`.
- `add_lesson(scenario, error, root_cause, fix)` → adds one error→fix to `error_feedback`.

In [ ]:
# collections.add_template(
#     instruction='Place an interdigitized current mirror on gf180',
#     code='component = current_mirror(gf180_mapped_pdk, numcols=3, device="nfet")',
#     circuit='current_mirror')
# collections.add_lesson(
#     scenario='met2 routes too close on gf180', error='M2.2a spacing < 0.28um',
#     root_cause='util_max_metal_seperation was sky130 0.3um', fix='raise to 0.5')
print('add_template / add_lesson ready')

## 4. Add a NEW data source / rebuild everything

To add a brand-new corpus: drop files under `data/rf_theory/` (texts/ jsonl) or add
circuits to `data/visual_dataset/dataset.jsonl`, then re-run the matching `ingest_*`.
`build_all` does all three at once (collection 3 starts empty by design).

In [ ]:
# print(collections.build_all(parse_pdfs=False, seed_feedback=False))
print('build_all ready — rebuilds glayout_knowledge + rf_theory, empties error_feedback')

## What is the "session DB" (`outputs/kaizen/research_db/`)?

It is a **separate, throwaway ChromaDB** the *Researcher* builds per design job: when
the agent web-searches papers/repos for a new request, it embeds those hits into a
**temporary** `temp_research_<jobid>` collection so that one run can retrieve them.
On success the useful bits are promoted into the permanent `rf_theory`
(`researcher.persist_on_success`); on failure they're dropped.

So it is **scratch space, not a knowledge base** — safe to delete between runs. It is
kept out of `data/chroma_db` on purpose so transient web junk never pollutes the
curated collections. (If you ever want a research result kept, it's already promoted
to `rf_theory` automatically.)